# Phase C Strict JAX Gate (Pinned Runtime Lock Manifest)

This notebook runs the pinned-manifest S8 flow and then executes `phase_c_gate.py` in strict mode.

Notes:
- On H100, set `DEVICE = "gpu"`.
- Keep `STRICT_JAX = True` and fallback disabled in gate execution.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

# Mirror the Kaggle bootstrap pattern from h100_validation_kaggle.ipynb
REPO_INPUT_DIR = Path('/kaggle/input/datasets/danielwuu/iris-main/packages')
WHEEL_DIR = Path('/kaggle/input/datasets/danielwuu/wheels/wheels/linux_cp312')
WORK_REPO = Path('/kaggle/working/IRIS')


def run_shell(cmd: str, cwd: Path | None = None, ok: tuple[int, ...] = (0,), env: dict | None = None):
    print(f'$ {cmd}')
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(proc.stdout)
    if proc.returncode not in ok:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')
    return proc


ROOT_CANDIDATES = [
    WORK_REPO,
    Path('/kaggle/working/iris-main'),
    REPO_INPUT_DIR,
    Path.cwd(),
]
ROOT = next((p for p in ROOT_CANDIDATES if (p / 'scripts').exists()), None)
if ROOT is None:
    raise FileNotFoundError(f'Cannot locate repo root from candidates: {ROOT_CANDIDATES}')

# /kaggle/input is read-only; copy repo to /kaggle/working before running scripts.
if str(ROOT).startswith('/kaggle/input/'):
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    shutil.copytree(ROOT, WORK_REPO)
    ROOT = WORK_REPO

PYTHON = Path(sys.executable)
DEVICE = 'gpu'
STRICT_JAX = True
BASELINE_REPORT_DIR = ROOT / 'artifacts' / 'phase_c_gate_20260301' / 'report'
PHASE_ROOT = ROOT / 'artifacts' / 'phase_c_gate_20260301'
PINNED_MANIFEST = ROOT / 'artifacts' / 'runtime_lock_pinned' / 'runtime_lock_manifest.json'

if WHEEL_DIR.exists():
    run_shell(f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} --upgrade pip")

    def wheel_version(path: Path) -> str:
        return path.name.split('-')[1]

    jax_whls = sorted(WHEEL_DIR.glob('jax-*.whl'))
    jaxlib_whls = sorted(WHEEL_DIR.glob('jaxlib-*.whl'))
    if not jax_whls or not jaxlib_whls:
        raise RuntimeError(f'Missing jax/jaxlib wheels in {WHEEL_DIR}')

    jax_by_version = {wheel_version(p): p for p in jax_whls}
    jaxlib_by_version = {wheel_version(p): p for p in jaxlib_whls}
    common_versions = sorted(set(jax_by_version) & set(jaxlib_by_version))
    if not common_versions:
        raise RuntimeError(f'No common jax/jaxlib versions in {WHEEL_DIR}')

    target_version = common_versions[-1]
    wheels_to_install = [jax_by_version[target_version], jaxlib_by_version[target_version]]

    plugin_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_plugin-*.whl'))}
    pjrt_by_version = {wheel_version(p): p for p in sorted(WHEEL_DIR.glob('jax_cuda12_pjrt-*.whl'))}
    if target_version in plugin_by_version and target_version in pjrt_by_version:
        wheels_to_install.extend([plugin_by_version[target_version], pjrt_by_version[target_version]])
    else:
        run_shell(f"{sys.executable} -m pip uninstall -y jax_cuda12_plugin jax_cuda12_pjrt", ok=(0, 1))

    wheel_args = ' '.join(str(p) for p in wheels_to_install)
    run_shell(
        f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} "
        f"--upgrade --force-reinstall --no-deps {wheel_args}"
    )
    run_shell(
        f"{sys.executable} -m pip install --no-index --find-links {WHEEL_DIR} "
        "--upgrade --force-reinstall numpy pytest flax optax orbax-checkpoint ml-dtypes scipy"
    )
else:
    print('WHEEL_DIR not found, skipping offline wheel install:', WHEEL_DIR)

RUN_ENV = os.environ.copy()
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
RUN_ENV.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.85')

import jax

print('python:', PYTHON)
print('repo root:', ROOT)
print('jax version:', jax.__version__)
print('jax devices:', jax.devices())
if DEVICE == 'gpu' and not any(dev.platform == 'gpu' for dev in jax.devices()):
    raise RuntimeError('No JAX GPU device detected on this runtime.')

ROOT


In [ ]:
def run(args, check=True):
    cmd = [str(PYTHON)] + args
    print('\n$ ' + ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, env=RUN_ENV)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed rc={proc.returncode}')
    return proc


def strict_jax_flags():
    return ['--device', DEVICE] + (['--strict-jax'] if STRICT_JAX else ['--no-strict-jax'])


def run_crash_resume(output_dir: Path, crash_point: str, resume_path_id: str):
    shutil.rmtree(output_dir, ignore_errors=True)
    run([
        'scripts/train_toy.py',
        '--output-dir', str(output_dir),
        '--segments', '1',
        '--crash-point', crash_point,
        '--crash-segment', '0',
        '--runtime-lock-manifest', str(PINNED_MANIFEST),
    ] + strict_jax_flags(), check=False)
    run([
        'scripts/train_toy.py',
        '--output-dir', str(output_dir),
        '--segments', '1',
        '--resume-path-id', resume_path_id,
        '--runtime-lock-manifest', str(PINNED_MANIFEST),
    ] + strict_jax_flags())


In [ ]:
# 1) Bootstrap one pinned manifest (single source of truth)
seed_dir = ROOT / 'artifacts' / 'runtime_lock_seed'
shutil.rmtree(seed_dir, ignore_errors=True)
run([
    'scripts/train_toy.py',
    '--output-dir', str(seed_dir),
    '--segments', '0',
] + strict_jax_flags())

PINNED_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(seed_dir / 'runtime_lock_manifest.json', PINNED_MANIFEST)
print('Pinned manifest:', PINNED_MANIFEST)

In [ ]:
# 2) Re-run S1/S2/S2M with strict JAX path
run(['scripts/s1_smoke.py', '--device', DEVICE])
run(['scripts/s2_structural.py'])
run(['scripts/s2_mounted.py', '--device', DEVICE])

In [ ]:
# 3) Rebuild local S8 packet using pinned manifest
for name in ['s8_uninterrupted', 's8_execute', 's8_pre_commit', 's8_post_commit']:
    shutil.rmtree(PHASE_ROOT / name, ignore_errors=True)

run([
    'scripts/train_toy.py',
    '--output-dir', str(PHASE_ROOT / 's8_uninterrupted'),
    '--segments', '1',
    '--resume-path-id', 'uninterrupted',
    '--runtime-lock-manifest', str(PINNED_MANIFEST),
] + strict_jax_flags())
run_crash_resume(PHASE_ROOT / 's8_execute', 'execute', 'execute_crash')
run_crash_resume(PHASE_ROOT / 's8_pre_commit', 'pre_commit', 'pre_commit_crash')
run_crash_resume(PHASE_ROOT / 's8_post_commit', 'post_commit', 'post_commit_crash')

In [ ]:
# 4) Rebuild H100 packet inputs using the same pinned manifest
h100_uninterrupted = ROOT / 'artifacts' / 'toy_train_gpu_h100'
h100_execute = ROOT / 'artifacts' / 'toy_resume_h100'
h100_pre = ROOT / 'artifacts' / 'toy_resume_h100_pre_commit'
h100_post = ROOT / 'artifacts' / 'toy_resume_h100_post_commit'

for path in [h100_uninterrupted, h100_execute, h100_pre, h100_post]:
    shutil.rmtree(path, ignore_errors=True)

run([
    'scripts/train_toy.py',
    '--output-dir', str(h100_uninterrupted),
    '--segments', '1',
    '--resume-path-id', 'uninterrupted',
    '--runtime-lock-manifest', str(PINNED_MANIFEST),
] + strict_jax_flags())
run_crash_resume(h100_execute, 'execute', 'execute_crash')
run_crash_resume(h100_pre, 'pre_commit', 'pre_commit_crash')
run_crash_resume(h100_post, 'post_commit', 'post_commit_crash')

In [ ]:
# 5) Strict gate run (fallback disabled, baseline required for S4/S5)
gate = run([
    'scripts/phase_c_gate.py',
    '--phase-root', str(PHASE_ROOT),
    '--output-dir', str(BASELINE_REPORT_DIR),
    '--baseline-report-dir', str(BASELINE_REPORT_DIR),
    '--strict-suite-exec',
    '--no-reuse-existing-suite-artifacts',
    '--metric-epsilon', '1e-5',
], check=False)
print('phase_c_gate rc =', gate.returncode)

In [ ]:
# 6) Inspect summary
summary_path = BASELINE_REPORT_DIR / 'summary_report.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('regression.status =', summary.get('regression.status'))
print('suite_status =', json.dumps(summary.get('suite_status', {}), indent=2, ensure_ascii=False))
print('violations =', json.dumps(summary.get('regression.violations', []), indent=2, ensure_ascii=False))